# Clase 076 — Calibración: Platt (sigmoid) e isotonic

GBM out-of-the-box suele estar mal calibrado. Reliability curve + Brier + ECE, después `CalibratedClassifierCV` con `sigmoid` e `isotonic`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, log_loss

np.random.seed(42)

## 1. Dataset + GBM

In [ ]:
X, y = make_classification(n_samples=8000, n_features=20, n_informative=10,
                            n_redundant=5, weights=[0.6, 0.4], random_state=42)
X_train, X_holdout, y_train, y_holdout = train_test_split(X, y, test_size=0.5, stratify=y, random_state=42)
X_cal, X_test, y_cal, y_test = train_test_split(X_holdout, y_holdout, test_size=0.5,
                                                   stratify=y_holdout, random_state=42)
print('train', X_train.shape, '| cal', X_cal.shape, '| test', X_test.shape)

gbm = GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42)
gbm.fit(X_train, y_train)
proba_raw = gbm.predict_proba(X_test)[:, 1]

## 2. Reliability curve manual (binning)

In [ ]:
def reliability_curve(y_true, proba, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    bin_idx = np.digitize(proba, bins) - 1
    bin_idx = np.clip(bin_idx, 0, n_bins - 1)
    out = []
    for b in range(n_bins):
        mask = bin_idx == b
        if mask.sum() == 0:
            continue
        out.append({
            'bin': b,
            'mean_pred': proba[mask].mean(),
            'frac_pos':  y_true[mask].mean(),
            'count':     int(mask.sum()),
        })
    return pd.DataFrame(out)

rel_raw = reliability_curve(y_test, proba_raw)
print(rel_raw.round(4).to_string(index=False))

## 3. Brier, log-loss, ECE

In [ ]:
def ece(y_true, proba, n_bins=10):
    df = reliability_curve(y_true, proba, n_bins)
    n = len(y_true)
    return float(((df['count'] / n) * (df['mean_pred'] - df['frac_pos']).abs()).sum())

def report(name, y_true, proba):
    return {
        'modelo': name,
        'brier':    brier_score_loss(y_true, proba),
        'log_loss': log_loss(y_true, np.clip(proba, 1e-6, 1 - 1e-6)),
        'ECE':      ece(y_true, proba),
    }

metrics = [report('GBM raw', y_test, proba_raw)]
print(pd.DataFrame(metrics).round(4).to_string(index=False))

## 4. Calibración con Platt (sigmoid) e isotonic

Usamos `cv='prefit'` con un set de calibración held-out.

In [ ]:
cal_sig = CalibratedClassifierCV(gbm, method='sigmoid', cv='prefit').fit(X_cal, y_cal)
cal_iso = CalibratedClassifierCV(gbm, method='isotonic', cv='prefit').fit(X_cal, y_cal)

proba_sig = cal_sig.predict_proba(X_test)[:, 1]
proba_iso = cal_iso.predict_proba(X_test)[:, 1]

metrics = [
    report('GBM raw', y_test, proba_raw),
    report('GBM + Platt (sigmoid)', y_test, proba_sig),
    report('GBM + isotonic', y_test, proba_iso),
]
print(pd.DataFrame(metrics).round(4).to_string(index=False))

## 5. Plot de calibración

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], 'k--', label='perfecto')
for name, p in [('GBM raw', proba_raw), ('Platt', proba_sig), ('Isotonic', proba_iso)]:
    rc = reliability_curve(y_test, p)
    ax.plot(rc['mean_pred'], rc['frac_pos'], marker='o', label=name)
ax.set_xlabel('predicción promedio por bin')
ax.set_ylabel('fracción positiva real')
ax.set_title('Reliability curve')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Histograma de probabilidades

Visualiza cómo cada método redistribuye los scores.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, name, p in zip(axes, ['raw', 'Platt', 'Isotonic'], [proba_raw, proba_sig, proba_iso]):
    ax.hist(p, bins=20, color='#37a', edgecolor='white')
    ax.set_title(name)
    ax.set_xlabel('proba predicha')
axes[0].set_ylabel('count')
plt.tight_layout()
plt.show()

## 7. Cuándo usar cada uno

- **sigmoid (Platt)**: pocas muestras de calibración (< ~1000) y miscalibración monotónica — dos parámetros, robusto.
- **isotonic**: más datos disponibles; no asume forma — más flexible pero puede sobreajustar.

## Ejercicios

1. Reducí `X_cal` a 200 ejemplos y volé a comparar Brier. ¿Qué método aguanta mejor?
2. Aplicá calibración a `RandomForestClassifier` (clásicamente mal calibrado).
3. Implementá temperature scaling para multiclase (`softmax(z/T)`).

## Conclusiones

- Accuracy no mide calibración — usá Brier y ECE.
- `CalibratedClassifierCV(cv='prefit')` requiere un set held-out exclusivo para evitar leak.
- En límite de datos, Platt es la apuesta conservadora; con muchos datos, isotonic gana.